# Pet Translator - Llama 3.2 LoRA Training
This notebook installs Unsloth, trains Llama 3.2 1B on your synthetic dataset, and exports the GGUF file for local Docker inference.

In [ ]:
# 1. Install Unsloth and dependencies
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
# 2. Download the synthetic dataset from the GitHub repository
import os
!wget -q https://raw.githubusercontent.com/MaayZz/pet_translator/main/training/llm/synthetic_dataset.json -O synthetic_dataset.json
print("Dataset downloaded!")

In [ ]:
# 3. Load Llama 3.2 Model
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
# 4. Load & Format Dataset
from datasets import load_dataset
dataset = load_dataset("json", data_files="synthetic_dataset.json", split="train")

def format_prompts(examples):
    instructions = examples["instruction"]
    outputs = examples["output"]
    texts = []
    for inst, out in zip(instructions, outputs):
        text = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nPet Translator<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n{inst}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n{out}<|eot_id|>"
        texts.append(text)
    return { "text" : texts }

dataset = dataset.map(format_prompts, batched=True)

In [ ]:
# 5. Train the Model
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Change to a higher number for real training (e.g. 300)
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)
trainer_stats = trainer.train()

In [ ]:
# 6. Export to GGUF format
print("Exporting to GGUF... This will take a few minutes.")
model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")

In [ ]:
# 7. Download to local machine
from google.colab import files
files.download("model-unsloth.Q4_K_M.gguf")
print("Rename this downloaded file to 'unsloth.Q4_K_M.gguf' and place it in your local 'backend-abir/models/' folder.")